In [ ]:
# Check for genome version consistency in the GWAS and apaQTL data. Liftover the coordinates of GRCh37 to GRCh38
!pip install liftover pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Convert the GRCh37 GWAS coordinates to GRCh38
# SNPs thst do not exist in GRCh38 will be dropped

import pandas as pd
from liftover import get_lifter
import ssl

# --- Configuration ---
INPUT_GWAS = r'.tsv'       
OUTPUT_GWAS = r'tsv'      

EXPECTED_COLUMNS = [
    'chromosome', 'base_pair_location', 'effect_allele', 'other_allele', 
    'beta', 'standard_error', 'effect_allele_frequency', 'p_value', 'rsid', 'info', 'effect_allele_frequency_cases', 'n', 'ncases', 
    'ncontrols', 'heterogeneity_i2', 'heterogeneity_df', 'heterogeneity_p_value' ]

def perform_liftover():
    print("Initializing hg19 to hg38 converter...")
    converter = get_lifter('hg19', 'hg38')

    chunk_size = 500000
    first_chunk = True
    total_processed = 0
    total_dropped = 0

    print("Starting liftover process...")

    for chunk in pd.read_csv(INPUT_GWAS, sep='\t', chunksize=chunk_size, dtype={'chromosome': str}):
        
        # Convert '23' to 'X'
        chunk['chromosome'] = chunk['chromosome'].replace('23', 'X')
        
        # Helper function to convert a single row's coordinate
        def lift_coordinate(row):
            chrom = row['chromosome']
            if not chrom.startswith('chr'):
                chrom = f"chr{chrom}"
            pos = row['base_pair_location']
            
            new_coords = converter.query(chrom, pos)
            if new_coords:
                return new_coords[0][1] 
            else:
                return None

        # 1. Apply the hg38 conversion
        chunk['bp_hg38'] = chunk.apply(lift_coordinate, axis=1)

        # 2. Count and drop unmapped SNPs
        unmapped_count = chunk['bp_hg38'].isna().sum()
        total_dropped += unmapped_count
        chunk = chunk.dropna(subset=['bp_hg38'])
        
        # 3. Overwrite the old location with the new hg38 location
        chunk['base_pair_location'] = chunk['bp_hg38'].astype(int)
        
        # 5. Clean up and strictly enforce the original column order
        chunk = chunk[EXPECTED_COLUMNS]

        # 6. Save the updated chunk
        chunk.to_csv(
            OUTPUT_GWAS, 
            sep='\t', 
            index=False, 
            mode='w' if first_chunk else 'a', 
            header=first_chunk
        )
        
        first_chunk = False
        total_processed += len(chunk)
        print(f"Processed {total_processed} successfully mapped SNPs...")

    print(f"\nLiftover Complete!")
    print(f"Successfully saved to: {OUTPUT_GWAS}")
    print(f"Dropped {total_dropped} SNPs that do not exist in GRCh38.")

if __name__ == "__main__":
    perform_liftover()

Initializing hg19 to hg38 converter...
Starting liftover process...
Processed 499974 successfully mapped SNPs...
Processed 999941 successfully mapped SNPs...
Processed 1499934 successfully mapped SNPs...
Processed 1999925 successfully mapped SNPs...
Processed 2499923 successfully mapped SNPs...
Processed 2999896 successfully mapped SNPs...
Processed 3499863 successfully mapped SNPs...
Processed 3999822 successfully mapped SNPs...
Processed 4499812 successfully mapped SNPs...
Processed 4999794 successfully mapped SNPs...
Processed 5499788 successfully mapped SNPs...
Processed 5999688 successfully mapped SNPs...
Processed 6499664 successfully mapped SNPs...
Processed 6999617 successfully mapped SNPs...
Processed 7362898 successfully mapped SNPs...

Liftover Complete!
Successfully saved to: D:\Project_CompBio\Summer_2026\QTL_Project\GWAS\Major Depressive Disorder\Sex-combined GWAS\MDD_sex_combined_GWAS_hg38.tsv
Dropped 404 SNPs that do not exist in GRCh38.


In [ ]:
# Obtain the snps that are in the 2Mb (+1Mb and -1Mb) region around the significant GWAS SNPs

import pandas as pd
import os

# --- Configuration ---
CSV_FILE_PATH = r'.csv'  
TSV_FILE_PATH = r'.tsv' 
WINDOW_SIZE = 1000000 #(the window size is 2Mb)

def extract_gwas_windows():
    print("Loading datasets into memory...")
    
    # 1. Load the files
    df_csv = pd.read_csv(CSV_FILE_PATH)
    df_tsv = pd.read_csv(TSV_FILE_PATH, sep='\t') 

    # Ensure BP column is numeric
    df_tsv['base_pair_location'] = pd.to_numeric(df_tsv['base_pair_location'], errors='coerce')

    # Detect the exact name of the chromosome column in the TSV
    chrom_col = None
    for col in ['chromosome', 'CHR', 'chr', 'Chromosome']:
        if col in df_tsv.columns:
            chrom_col = col
            break
            
    if not chrom_col:
        print("Error: Could not find a chromosome column in the TSV file.")
        return

    # Extract target rs_ids from the CSV
    target_rsids = df_csv['rs_id'].dropna().unique()
    print(f"Found {len(target_rsids)} unique lead SNPs to process.\n")

    # 2. Iterate and Extract
    for rsid in target_rsids:
        # Find the lead SNP inside the TSV
        lead_snp_row = df_tsv[df_tsv['rs_id'] == rsid]

        if lead_snp_row.empty:
            print(f"Warning: Lead SNP {rsid} not found in the GWAS TSV. Skipping.")
            continue

        # --- THE CRITICAL STEP ---
        # Extract both the Base Pair AND the Chromosome directly from the TSV
        lead_bp = lead_snp_row['base_pair_location'].iloc[0]
        lead_chrom = lead_snp_row[chrom_col].iloc[0] 
        
        # Calculate boundaries
        lower_bound = lead_bp - WINDOW_SIZE
        upper_bound = lead_bp + WINDOW_SIZE

        # Filter the TSV using the TSV's own chromosome and boundaries
        window_df = df_tsv[
            (df_tsv[chrom_col] == lead_chrom) &
            (df_tsv['base_pair_location'] >= lower_bound) &
            (df_tsv['base_pair_location'] <= upper_bound)
        ]

        # Save the file
        output_filename = f"{rsid}_2mb_GWAS_SNPs_male.tsv"
        window_df.to_csv(output_filename, sep='\t', index=False)
        
        print(f"Saved {len(window_df)} SNPs for {rsid} (Chr {lead_chrom}) -> {output_filename}")

    print("\nExtraction complete!")

if __name__ == "__main__":
    extract_gwas_windows()

Loading datasets into memory...
Found 8 unique lead SNPs to process.

Saved 4369 SNPs for rs11209943 (Chr 1) -> rs11209943_2mb_GWAS_SNPs_male.tsv
Saved 5906 SNPs for rs11688767 (Chr 2) -> rs11688767_2mb_GWAS_SNPs_male.tsv
Saved 5137 SNPs for rs17782683 (Chr 14) -> rs17782683_2mb_GWAS_SNPs_male.tsv
Saved 5185 SNPs for rs2268978 (Chr 14) -> rs2268978_2mb_GWAS_SNPs_male.tsv
Saved 6920 SNPs for rs2298969 (Chr 4) -> rs2298969_2mb_GWAS_SNPs_male.tsv
Saved 4032 SNPs for rs4348675 (Chr 1) -> rs4348675_2mb_GWAS_SNPs_male.tsv
Saved 5601 SNPs for rs4547421 (Chr 18) -> rs4547421_2mb_GWAS_SNPs_male.tsv
Saved 3305 SNPs for rs5971319 (Chr X) -> rs5971319_2mb_GWAS_SNPs_male.tsv

Extraction complete!


In [ ]:
# Create a csv file which contains the coordinates of the start and end of the 2Mb regions for the LD matirx

import pandas as pd

def create_ld_regions(input_tsv, output_csv):
    # Read the input TSV
    df = pd.read_csv(input_tsv, sep='\t')
    
    # Ensure the required columns exist
    if not {'chromosome', 'base_pair_location', 'rs_id'}.issubset(df.columns):
        raise ValueError("The TSV must contain 'chromosome' and 'base_pair_location' columns.")
    
    # Create the new columns (+/- 1 million bp)
    df['from-bp'] = df['base_pair_location'] - 1000000
    df['to-bp'] = df['base_pair_location'] + 1000000
    
    # Base pair positions cannot be negative or zero, so we floor it at 1
    df['from-bp'] = df['from-bp'].clip(lower=1)
    
    # Select and order the final columns
    final_df = df[['rs_id', 'chromosome', 'base_pair_location', 'from-bp', 'to-bp']]
    
    # Export to CSV without the index column
    final_df.to_csv(output_csv, index=False)
    print(f"Successfully generated: {output_csv}")


create_ld_regions('female_sig_gwas_snps_hg38.tsv', 'female_sig_gwas_snps_2mb_window_hg38.csv')

Successfully generated: female_sig_gwas_snps_2mb_window_hg38.csv


In [ ]:
# Print the number of significant snps (pvalue < 5e-8) in the 2Mb window

import os
import glob
import pandas as pd

# 1. Define the path to your folder
folder_path = r"D:\Project_CompBio\Summer_2026\QTL_Project\GWAS\Major Depressive Disorder\Sex-stratified GWAS\2mb_GWAS_SNPs_female"

# 2. Define the search pattern for your specific files
# The '*' acts as a wildcard for the {snp} part of your filename
search_pattern = os.path.join(folder_path, "*_2mb_GWAS_SNPs_female.tsv")

# 3. Find all files matching the pattern
target_files = glob.glob(search_pattern)

print(f"Found {len(target_files)} matching files. Starting analysis...\n")

# 4. Loop through each file
for file_path in target_files:
    # Extract just the filename from the full path for cleaner printing
    filename = os.path.basename(file_path)
    
    try:
        # Read the TSV file into a pandas DataFrame
        df = pd.read_csv(file_path, sep='\t')
        
        # Check if the 'p_value' column actually exists to prevent crashes
        if 'p_value' in df.columns:
            
            # Count how many rows have a p_value strictly less than 5*10^-8
            # .sum() works because True evaluates to 1 and False evaluates to 0
            sig_count = (df['p_value'] < 5e-8).sum()
            
            print(f"{filename}: {sig_count} significant SNPs (p < 5e-8)")
            
        else:
            print(f"{filename}: WARNING - 'p_value' column not found in this file.")
            
    except Exception as e:
        # If the file is corrupted or completely empty, this prevents the loop from dying
        print(f"{filename}: ERROR reading file - {e}")

print("\nFinished processing all files.")

Found 16 matching files. Starting analysis...

rs10123941_2mb_GWAS_SNPs_female.tsv: 28 significant SNPs (p < 5e-8)
rs10502971_2mb_GWAS_SNPs_female.tsv: 33 significant SNPs (p < 5e-8)
rs11130182_2mb_GWAS_SNPs_female.tsv: 4 significant SNPs (p < 5e-8)
rs1142828_2mb_GWAS_SNPs_female.tsv: 1 significant SNPs (p < 5e-8)
rs11509880_2mb_GWAS_SNPs_female.tsv: 97 significant SNPs (p < 5e-8)
rs12134194_2mb_GWAS_SNPs_female.tsv: 838 significant SNPs (p < 5e-8)
rs127382_2mb_GWAS_SNPs_female.tsv: 18 significant SNPs (p < 5e-8)
rs13043844_2mb_GWAS_SNPs_female.tsv: 158 significant SNPs (p < 5e-8)
rs1443918_2mb_GWAS_SNPs_female.tsv: 2 significant SNPs (p < 5e-8)
rs1931263_2mb_GWAS_SNPs_female.tsv: 2 significant SNPs (p < 5e-8)
rs2963222_2mb_GWAS_SNPs_female.tsv: 143 significant SNPs (p < 5e-8)
rs4350429_2mb_GWAS_SNPs_female.tsv: 2 significant SNPs (p < 5e-8)
rs4456268_2mb_GWAS_SNPs_female.tsv: 5 significant SNPs (p < 5e-8)
rs4788616_2mb_GWAS_SNPs_female.tsv: 11 significant SNPs (p < 5e-8)
rs56113727_2m

In [ ]:
# Obtain the apaQTLs for each tissue in the 2Mb region around each significant GWAS SNP

import polars as pl
import os
import glob
import re
from functools import reduce
import operator

# --- Configuration ---
TSV_FILE_PATH = r'.tsv'
PARQUET_FOLDER = r''
WINDOW_SIZE = 1_000_000

def extract_apaqtl_windows():
    print("Loading target SNPs...")
    
    # 1. Load the target SNPs
    try:
        df_snps = pl.read_csv(TSV_FILE_PATH, separator='\t')
    except Exception as e:
        print(f"Error loading TSV: {e}")
        return

    # 2. Get unique chromosomes to process them one by one
    chromosomes = df_snps.select("chromosome").unique().to_series().to_list()
    
    for chrom in chromosomes:
        # Normalize chromosome string (ensure it doesn't have 'chr' prefix to match filename)
        chrom_str = str(chrom).replace("chr", "")
        
        # Get all SNPs on this specific chromosome
        chrom_snps = df_snps.filter(pl.col("chromosome") == chrom)
        
        # Build the exact search pattern for the parquet files
        # e.g., Brain_*.v10.cis_apaqtl.allpairs.chr1.parquet
        pattern = os.path.join(PARQUET_FOLDER, f"Brain_*.v10.cis_apaqtl.allpairs.chr{chrom_str}.parquet")
        parquet_files = glob.glob(pattern)
        
        if not parquet_files:
            print(f"Warning: No parquet files found for chromosome {chrom_str}.")
            continue
            
        print(f"\n--- Processing Chromosome {chrom_str} ({len(chrom_snps)} SNPs) ---")
        
        # Prepare the window ranges for all SNPs on this chromosome
        snp_windows = []
        for row in chrom_snps.iter_rows(named=True):
            rs_id = row['rs_id']
            bp = row['base_pair_location']
            lower_bound = bp - WINDOW_SIZE
            upper_bound = bp + WINDOW_SIZE
            snp_windows.append((rs_id, lower_bound, upper_bound))
        
        # 3. Iterate through each tissue parquet file for this chromosome
        for pq_file in parquet_files:
            # Extract tissue name using regex
            match = re.search(r"Brain_(.+?)\.v10", os.path.basename(pq_file))
            if not match:
                continue
            tissue = match.group(1)
            print(f"Scanning Brain_{tissue}...")

            # Open a LAZY scan of the massive parquet file (does not load into RAM yet)
            lf = pl.scan_parquet(pq_file)
            
            # Create a dynamic column that extracts the position from the variant_id string
            # variant_id format: chr1_10500_A_T_b38 -> splitting by '_' and taking index 1
            lf = lf.with_columns(
                pl.col("variant_id").str.split("_").list.get(1).cast(pl.Int64).alias("extracted_pos")
            )
            
            # Build a combined OR filter for ALL SNPs on this chromosome
            # This ensures we only scan the massive parquet file ONE time.
            conditions = [
                (pl.col("extracted_pos") >= lower) & (pl.col("extracted_pos") <= upper)
                for _, lower, upper in snp_windows
            ]
            combined_filter = reduce(operator.or_, conditions)
            
            # Execute the query: Filter the massive file and collect ONLY the matching rows into RAM
            try:
                df_filtered = lf.filter(combined_filter).collect()
            except Exception as e:
                print(f"  Error reading {os.path.basename(pq_file)}: {e}")
                continue
            
            if len(df_filtered) == 0:
                print(f"  No SNPs found in the 2Mb windows for Brain_{tissue}.")
                continue
                
            # 4. Split the loaded data by rs_id and save to individual TSV files
            for rs_id, lower, upper in snp_windows:
                # Isolate the data for this specific rs_id
                subset = df_filtered.filter(
                    (pl.col("extracted_pos") >= lower) & (pl.col("extracted_pos") <= upper)
                )
                
                if len(subset) > 0:
                    # Drop the temporary position column we created
                    subset = subset.drop("extracted_pos")
                    
                    output_filename = f"{rs_id}_2mb_Brain_{tissue}_apaQTL.tsv"
                    subset.write_csv(output_filename, separator='\t')
                    print(f"  -> Saved {len(subset)} rows to {output_filename}")

    print("\nExtraction Complete!")

if __name__ == "__main__":
    extract_apaqtl_windows()